In [15]:
import pandas as pd
from pathlib import Path

# -------------------------------------------------
# PATHS
# -------------------------------------------------
INPUT_DIR = Path("/Users/jakobwerkgarner/code/mt_dsnow/par_sens/SNOWPACK_data/data<2000m/raw_alpsolut")
OUTPUT_DIR = Path("/Users/jakobwerkgarner/code/mt_dsnow/par_sens/SNOWPACK_data/data<2000m/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# variables of interest
VARS = ["HS_mod", "HS_meas", "SWE"]


# -------------------------------------------------
# FUNCTIONS
# -------------------------------------------------
def read_smet(file_path):
    """Read SMET file into header and DataFrame."""
    header = []
    fields = None
    data_rows = []
    data_start = False

    with open(file_path, "r") as f:
        for line in f:
            line = line.rstrip()

            if line.startswith("fields"):
                fields = line.split("=", 1)[1].strip().split()

            elif line.strip() == "[DATA]":
                data_start = True
                continue

            elif not data_start:
                header.append(line)

            else:
                if line:
                    data_rows.append(line.split())

    df = pd.DataFrame(data_rows, columns=fields)
    df["timestamp"] = pd.to_datetime(df["timestamp"])

    return header, df


def extract_station_id(header):
    for line in header:
        if line.startswith("station_id"):
            return line.split("=", 1)[1].strip()
    raise ValueError("station_id not found in SMET header")


def snow_season(ts):
    """Snow season Nov–Apr → 'YYYY_YYYY'."""
    if ts.month >= 11:
        return f"{ts.year}_{ts.year + 1}"
    else:
        return f"{ts.year - 1}_{ts.year}"


# -------------------------------------------------
# MAIN LOOP
# -------------------------------------------------
for smet_file in INPUT_DIR.glob("*.smet"):
    print(f"Processing {smet_file.name}")

    header, df = read_smet(smet_file)
    station_id = extract_station_id(header)

    # keep only required variables
    missing = [v for v in VARS if v not in df.columns]
    if missing:
        raise ValueError(f"{smet_file.name} missing variables: {missing}")

    df = df[["timestamp"] + VARS]

    # force numeric, convert nodata to NaN
    df[VARS] = (
        df[VARS]
        .apply(pd.to_numeric, errors="coerce")
        .replace(-999, pd.NA)
    )

    df["station_id"] = station_id
    df["season"] = df["timestamp"].apply(snow_season)

    for season, df_season in df.groupby("season"):
        start_year = int(season.split("_")[0])

        start = pd.Timestamp(start_year, 10, 1)
        end = pd.Timestamp(start_year + 1, 4, 30, 23, 59)

        df_season = df_season[
            (df_season["timestamp"] >= start) &
            (df_season["timestamp"] <= end)
        ]

        if df_season.empty:
            continue

        # -------------------------------------------------
        # DAILY MEAN
        # -------------------------------------------------
        df_daily = (
            df_season
            .set_index("timestamp")[VARS]
            .resample("D")
            .mean()
            .reset_index()
        )

        df_daily["station_id"] = station_id
        df_daily["season"] = season

        out_file = OUTPUT_DIR / f"{station_id}_{season}_daily_HS_SWE.csv"
        df_daily.to_csv(out_file, index=False)

        print(f"  -> written {out_file.name}")

Processing smet_54_2021-07-31T22_00_00.000Z_2025-07-30T22_00_00.000Z_1778840953162.smet
  -> written NGAN2_2023_2024_daily_HS_SWE.csv
  -> written NGAN2_2024_2025_daily_HS_SWE.csv
Processing smet_126_2021-07-31T22_00_00.000Z_2025-07-30T22_00_00.000Z_1778837888198.smet
  -> written PBRO2_2023_2024_daily_HS_SWE.csv
  -> written PBRO2_2024_2025_daily_HS_SWE.csv
Processing smet_45_2021-07-31T22_00_00.000Z_2025-07-30T22_00_00.000Z_1778838101230.smet
  -> written AXLIZ1_2023_2024_daily_HS_SWE.csv
  -> written AXLIZ1_2024_2025_daily_HS_SWE.csv
Processing smet_42_2021-07-31T22_00_00.000Z_2025-07-30T22_00_00.000Z_1778837783860.smet
  -> written BJOE2_2023_2024_daily_HS_SWE.csv
  -> written BJOE2_2024_2025_daily_HS_SWE.csv
Processing smet_82_2021-07-31T22_00_00.000Z_2025-07-30T22_00_00.000Z_1778837871982.smet
  -> written PMER2_2023_2024_daily_HS_SWE.csv
  -> written PMER2_2024_2025_daily_HS_SWE.csv
Processing smet_56_2021-07-31T22_00_00.000Z_2025-07-30T22_00_00.000Z_1778837758289.smet
  -> writ

In [16]:
# Build station metadata table from SMET headers and save to CSV
metadata_rows = []

for smet_file in INPUT_DIR.glob("*.smet"):
    header, _ = read_smet(smet_file)

    header_dict = {}
    for line in header:
        if "=" in line:
            key, value = line.split("=", 1)
            header_dict[key.strip()] = value.strip()

    metadata_rows.append(
        {
            "station_name": header_dict.get("station_name"),
            "station_id": header_dict.get("station_id"),
            "lat": pd.to_numeric(header_dict.get("latitude"), errors="coerce"),
            "lon": pd.to_numeric(header_dict.get("longitude"), errors="coerce"),
            "altitude": pd.to_numeric(header_dict.get("altitude"), errors="coerce"),
        }
    )

AAA_station_meate = (
    pd.DataFrame(metadata_rows)
    .drop_duplicates(subset=["station_id"])
    .sort_values("station_id")
    .reset_index(drop=True)
)

meta_out_file = OUTPUT_DIR / "AAA_station_meta.csv"
AAA_station_meate.to_csv(meta_out_file, index=False)

print(f"Written {meta_out_file}")
AAA_station_meate.head()

Written /Users/jakobwerkgarner/code/mt_dsnow/par_sens/SNOWPACK_data/data<2000m/processed/AAA_station_meta.csv


,station_name,station_id,lat,lon,altitude
0,Kleiner Beil,AKLE2,47.351489,12.016050,1880.0
1,Speicherteich Axamer Lizum,AXLIZ1,47.180105,11.288011,2103.0
2,Jochelspitze,BJOE2,47.273235,10.365900,1697.0
3,Nachtweide,INAC2,46.982623,10.306146,2030.0
4,Seegrube,ISEE2,47.306382,11.377934,1921.0
